# AdaOS Node In Google Colab

Этот блокнот поднимает AdaOS-ноду в Google Colab и хранит состояние `.adaos` на Google Drive.

Что делает блокнот:
- монтирует Google Drive
- хранит `ADAOS_BASE_DIR` на Drive
- клонирует или обновляет репозиторий `adaos`
- запускает `tools/bootstrap.sh` с параметрами `--root-url`, `--zone`, `--join-code`, `--node-name`
- стартует локальный API в фоне
- дает быстрые команды для статуса, логов и бэкапа

Примечание: runtime Colab временный, но состояние в `.adaos` сохраняется на Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# @title Parameters
ROOT_URL = "https://api.inimatic.com"  # @param {type:"string"}
ZONE = "ru"  # @param {type:"string"}
JOIN_CODE = ""  # @param {type:"string"}
NODE_NAME = "Colab Member"  # @param {type:"string"}
REV = "rev2026"  # @param {type:"string"}
REPO_URL = "https://github.com/stipot-com/adaos.git"  # @param {type:"string"}

DRIVE_ROOT = "/content/drive/MyDrive/adaos_colab"  # @param {type:"string"}
ADAOS_DRIVE_DIR = f"{DRIVE_ROOT}/.adaos"  # @param {type:"string"}
REPO_DIR = "/content/adaos"  # @param {type:"string"}
LOG_DIR = f"{DRIVE_ROOT}/logs"  # @param {type:"string"}

SERVE_HOST = "127.0.0.1"  # @param {type:"string"}
SERVE_PORT = 8777  # @param {type:"integer"}
INSTALL_SERVICE = "never"  # @param ["never", "auto"]
NO_CORE_UPDATE = True  # @param {type:"boolean"}

FORCE_RECLONE = False  # @param {type:"boolean"}
FORCE_CLEAN_VENV = False  # @param {type:"boolean"}

print({
    'ROOT_URL': ROOT_URL,
    'ZONE': ZONE,
    'JOIN_CODE_SET': bool(JOIN_CODE.strip()),
    'NODE_NAME': NODE_NAME,
    'REV': REV,
    'REPO_URL': REPO_URL,
    'ADAOS_DRIVE_DIR': ADAOS_DRIVE_DIR,
    'REPO_DIR': REPO_DIR,
    'LOG_DIR': LOG_DIR,
    'SERVE_HOST': SERVE_HOST,
    'SERVE_PORT': SERVE_PORT,
    'INSTALL_SERVICE': INSTALL_SERVICE,
    'NO_CORE_UPDATE': NO_CORE_UPDATE,
    'FORCE_RECLONE': FORCE_RECLONE,
    'FORCE_CLEAN_VENV': FORCE_CLEAN_VENV,
})

In [ ]:
from pathlib import Path
import os
import shutil

Path(DRIVE_ROOT).mkdir(parents=True, exist_ok=True)
Path(ADAOS_DRIVE_DIR).mkdir(parents=True, exist_ok=True)
Path(LOG_DIR).mkdir(parents=True, exist_ok=True)

if FORCE_RECLONE and Path(REPO_DIR).exists():
    shutil.rmtree(REPO_DIR)

print('Drive root:', DRIVE_ROOT)
print('AdaOS base dir:', ADAOS_DRIVE_DIR)
print('Repo dir:', REPO_DIR)
print('Log dir:', LOG_DIR)

In [ ]:
import os
import shlex
import subprocess
from pathlib import Path

repo = Path(REPO_DIR)
if not repo.exists():
    subprocess.run([
        'git', 'clone', '-b', REV, REPO_URL, REPO_DIR
    ], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', '--all', '--prune'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', REV], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)

if FORCE_CLEAN_VENV:
    venv_dir = repo / '.venv'
    if venv_dir.exists():
        subprocess.run(['rm', '-rf', str(venv_dir)], check=True)

print('Repo ready:', REPO_DIR)

In [ ]:
import os
import shlex
import subprocess

env = os.environ.copy()
env['ADAOS_BASE_DIR'] = ADAOS_DRIVE_DIR

cmd = [
    'bash', 'tools/bootstrap.sh',
    '--root-url', ROOT_URL,
    '--zone', ZONE,
    '--node-name', NODE_NAME,
    '--install-service', INSTALL_SERVICE,
    '--rev', REV,
]

if JOIN_CODE.strip():
    cmd += ['--join-code', JOIN_CODE.strip()]
if NO_CORE_UPDATE:
    cmd += ['--no-core-update']

print('Running:', ' '.join(shlex.quote(x) for x in cmd))
subprocess.run(cmd, cwd=REPO_DIR, env=env, check=True)

In [ ]:
import os
import shlex
import subprocess
from pathlib import Path

def run_adaos(*args, check=True):
    env = os.environ.copy()
    env['ADAOS_BASE_DIR'] = ADAOS_DRIVE_DIR
    quoted = ' '.join(shlex.quote(str(arg)) for arg in args)
    cmd = ['bash', '-lc', 'source .venv/bin/activate && python -m adaos ' + quoted]
    return subprocess.run(cmd, cwd=REPO_DIR, env=env, check=check)

def bash(script, check=True):
    env = os.environ.copy()
    env['ADAOS_BASE_DIR'] = ADAOS_DRIVE_DIR
    return subprocess.run(['bash', '-lc', script], cwd=REPO_DIR, env=env, check=check)

print('Helpers ready: run_adaos(...), bash(...)')

In [ ]:
run_adaos('where')
run_adaos('node', 'status', check=False)
bash(f'curl -sS http://{SERVE_HOST}:{SERVE_PORT}/api/node/status || true', check=False)

## Useful Operations

In [ ]:
# Start API in foreground
# run_adaos('api', 'serve', '--host', SERVE_HOST, '--port', str(SERVE_PORT), check=False)

# Stop API
# run_adaos('api', 'stop', check=False)

# Restart API
# run_adaos('api', 'restart', check=False)

In [ ]:
bash(f'curl -sS http://{SERVE_HOST}:{SERVE_PORT}/api/node/status || true', check=False)

In [ ]:
bash('find "$ADAOS_BASE_DIR" -maxdepth 2 -type f | sort | tail -n 80', check=False)

In [ ]:
from datetime import datetime
import os
import shlex

stamp = datetime.utcnow().strftime('%Y%m%d_%H%M%SZ')
snapshot = f'{LOG_DIR}/adaos_snapshot_{stamp}.tar.gz'
script = f'tar -czf {shlex.quote(snapshot)} -C {shlex.quote(DRIVE_ROOT)} .adaos'
bash(script)
print('Snapshot saved:', snapshot)

## Notes

- Если хочешь войти в существующую сеть как `member`, передай `JOIN_CODE`.
- Если `JOIN_CODE` пустой, bootstrap просто поднимет локальную ноду без `node join`.
- В Colab лучше оставлять `INSTALL_SERVICE="never"`, потому что systemd там обычно недоступен.
- Состояние ноды лежит в `ADAOS_DRIVE_DIR`, так что после переподключения Colab его можно переиспользовать.